# EDA — Subastas Públicas de Inmuebles en España
Fuente: Portal de Subastas del BOE (subastas.boe.es)<br>
Período analizado: Diciembre 2025

---
## Hipótesis
> *"La mayoría de los inmuebles subastados en el Portal del BOE 
> durante diciembre 2025 se ofertan por debajo del 50% de su valor 
> de tasación, lo que representa una oportunidad significativa de 
> compra para inversores y particulares"*

## Objetivo
Analizar el comportamiento de las subastas públicas de inmuebles 
en España: precios, descuentos sobre tasación, organismos 
convocantes y distribución geográfica.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('./utils')
from funciones import limpiar_dataframe, filtrar_tributarios, resumen_dataset

# Configuración de visualizaciones
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100
sns.set_theme(style='whitegrid')


# **1. VISUALIZACION**

### **1.1 Carga los 3 dataset de subastas (Octubre, noviembre y diciembre) que está en la ruta './data/'**

In [17]:
df_oct = pd.read_csv('./data/subastas_inmuebles_oct2025.csv')
df_nov = pd.read_csv('./data/subastas_inmuebles_nov2025.csv')
df_dic = pd.read_csv('./data/subastas_inmuebles_dic2025.csv')

print(f'Octubre   : {len(df_oct):,} registros')
print(f'Noviembre : {len(df_nov):,} registros')
print(f'Diciembre : {len(df_dic):,} registros')
print(f'Total     : {len(df_oct) + len(df_nov) + len(df_dic):,} registros')

Octubre   : 1,671 registros
Noviembre : 1,518 registros
Diciembre : 1,537 registros
Total     : 4,726 registros


### **1.2 Concatenar**

In [19]:
df = pd.concat([df_oct, df_nov, df_dic], ignore_index=True)

print(f'Dataset combinado: {df.shape}')
print(f'\n¿Hay duplicados?')
print(f'Filas duplicadas: {df.duplicated(subset="referencia").sum()}')

Dataset combinado: (4726, 23)

¿Hay duplicados?
Filas duplicadas: 0


In [21]:
#Primera 5 filas
df.head()

,referencia,num_lotes_listado,organismo,expediente,estado,fecha_conclusion_listado,descripcion,url_detalle,id_sub,referencia_det,...,fecha_conclusion,cantidad_reclamada_eur,lotes,anuncio_boe,valor_subasta_eur,tasacion_eur,puja_minima_eur,tramo_pujas_eur,deposito_eur,forma_adjudicacion
0,SUB-JA-2024-231148,1,JUZGADO 1 INSTANCIA 2 - LEON,0146/21,Pendiente de finalización y devolución de depó...,01/10/2025 a las 18:00:00,FINCA NÚMERO DIEZ.- Vivienda en la planta cuar...,https://subastas.boe.es/detalleSubasta.php?idS...,SUB-JA-2024-231148,SUB-JA-2024-231148,...,2025-10-01,38074.96,1,BOE-B-2025-31964,110887.98,0.0,NaN,NaN,5544.40,NaN
1,SUB-JA-2025-237883,1,UNIDAD SUBASTAS JUDICIALES MURCIA - MURCIA,0353/17,Finalizada y depósitos con reserva devueltos,01/10/2025 a las 18:00:00,Id. lote. 237883L1. Finca registral 37881. Pis...,https://subastas.boe.es/detalleSubasta.php?idS...,SUB-JA-2025-237883,SUB-JA-2025-237883,...,2025-10-01,56189.91,1,BOE-B-2025-31986,107904.68,0.0,NaN,2158.1,5395.23,NaN
2,SUB-JA-2025-242137,4,JUZGADO 1 INST E INSTRUCC. 1 - SOLSONA,0006/20,Finalizada y depósitos con reserva devueltos,01/10/2025 a las 18:00:00,"Subasta con varios lotes. Lote 1: URBANA, Parc...",https://subastas.boe.es/detalleSubasta.php?idS...,SUB-JA-2025-242137,SUB-JA-2025-242137,...,2025-10-01,112641.00,4,BOE-B-2025-31977,NaN,NaN,NaN,NaN,NaN,Separada para cada lote
3,SUB-JA-2025-244592,2,JUZGADO 1 INST E INSTRUCC. 1 - GANDESA,0442/20,Cancelada,NaN,Subasta con varios lotes. Lote 1: 1. URBANA. C...,https://subastas.boe.es/detalleSubasta.php?idS...,SUB-JA-2025-244592,SUB-JA-2025-244592,...,NaN,205737.80,2,BOE-B-2025-31954,NaN,NaN,NaN,NaN,NaN,Separada para cada lote
4,SUB-JA-2025-245672,1,Sección Civil e Instrucción TI Blanes. Plz.n 1...,0719/20,Finalizada y depósitos con reserva devueltos,01/10/2025 a las 18:00:00,URBANA: FINCA ESPECIAL. ENTIDAD NUMERO TRES.- ...,https://subastas.boe.es/detalleSubasta.php?idS...,SUB-JA-2025-245672,SUB-JA-2025-245672,...,2025-10-01,179629.82,1,BOE-B-2025-31945,259600.00,0.0,NaN,5192.0,12980.00,NaN


In [ ]:
#Tipos de datos
df.info() #=====================aqui me quede

<class 'pandas.DataFrame'>
RangeIndex: 4726 entries, 0 to 4725
Data columns (total 23 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   referencia                4726 non-null   str    
 1   num_lotes_listado         4726 non-null   int64  
 2   organismo                 4726 non-null   str    
 3   expediente                3360 non-null   str    
 4   estado                    4726 non-null   str    
 5   fecha_conclusion_listado  4364 non-null   str    
 6   descripcion               4653 non-null   str    
 7   url_detalle               4726 non-null   str    
 8   id_sub                    4726 non-null   str    
 9   referencia_det            4726 non-null   str    
 10  tipo_subasta              4726 non-null   str    
 11  cuenta_expediente         3360 non-null   str    
 12  fecha_inicio              4364 non-null   str    
 13  fecha_conclusion          4364 non-null   str    
 14  cantidad_reclamada_

In [14]:
#Estadísticas básicas
df.describe()

,num_lotes_listado,cantidad_reclamada_eur,lotes,valor_subasta_eur,tasacion_eur,puja_minima_eur,tramo_pujas_eur,deposito_eur
count,1537.000000,1.287000e+03,1537.000000,1.303000e+03,1.311000e+03,3.410000e+02,1094.000000,1311.000000
mean,1.903709,6.800985e+05,1.903709,2.684248e+05,1.171393e+05,6.668486e+04,4790.698300,13639.208375
std,5.227242,1.173695e+07,5.227242,9.595952e+05,7.018234e+05,4.429601e+05,17141.102073,42719.117400
min,1.000000,0.000000e+00,1.000000,2.449100e+02,0.000000e+00,1.000000e+00,8.100000,0.000000
25%,1.000000,4.002853e+04,1.000000,5.832501e+04,0.000000e+00,8.500000e+02,1000.000000,2963.975000
50%,1.000000,1.020086e+05,1.000000,1.314096e+05,0.000000e+00,5.294110e+03,2258.815000,6747.430000
75%,1.000000,1.897034e+05,1.000000,2.265382e+05,7.850007e+04,2.250000e+04,3905.807500,11876.400000
max,76.000000,4.175205e+08,76.000000,1.827946e+07,1.588523e+07,6.500000e+06,325754.000000,814385.000000


### **1.2 Analizando columnas duplicadas**

Durante la vista al dataset se observa que aparentemente hay columnas que tienen los mismos datos, sera necesario verificarlos.

In [7]:
# ====== Verificar las 3 columnas posiblemente iguales ('referencia', 'id_sub', 'referencia_det') =======
print('=== REFERENCIAS ===')
print(f'¿referencia == id_sub?')
print(f'Iguales     : {(df["referencia"] == df["id_sub"]).sum()}')
print(f'Distintos   : {(df["referencia"] != df["id_sub"]).sum()}')
print(f'Nulos id_sub: {df["id_sub"].isna().sum()}')

print("---------------------------------")
print(f'¿referencia == referencia_det?')
print(f'Iguales             : {(df["referencia"] == df["referencia_det"]).sum()}')
print(f'Distintos           : {(df["referencia"] != df["referencia_det"]).sum()}')
print(f'Nulos referencia_det: {df["referencia_det"].isna().sum()}')

print("---------------------------------")
# Ver ejemplos de los 3 juntos
print('Muestra de las 3 columnas:')
df[['referencia', 'id_sub', 'referencia_det']].head(5)

=== REFERENCIAS ===
¿referencia == id_sub?
Iguales     : 1537
Distintos   : 0
Nulos id_sub: 0
---------------------------------
¿referencia == referencia_det?
Iguales             : 1537
Distintos           : 0
Nulos referencia_det: 0
---------------------------------
Muestra de las 3 columnas:


,referencia,id_sub,referencia_det
0,SUB-RC-2025-3800100125018,SUB-RC-2025-3800100125018,SUB-RC-2025-3800100125018
1,SUB-AT-2025-24R4686001676,SUB-AT-2025-24R4686001676,SUB-AT-2025-24R4686001676
2,SUB-AT-2025-25R0886001398,SUB-AT-2025-25R0886001398,SUB-AT-2025-25R0886001398
3,SUB-AT-2025-25R0886001399,SUB-AT-2025-25R0886001399,SUB-AT-2025-25R0886001399
4,SUB-AT-2025-25R0886001455,SUB-AT-2025-25R0886001455,SUB-AT-2025-25R0886001455


In [8]:
# ====== Verificar las 2 columnas posiblemente iguales ('num_lotes_listado', 'lotes')=======

print('=== LOTES ===')
print(f'¿num_lotes_listado == lotes?')
iguales = (df['num_lotes_listado'] == df['lotes']).sum()
distintos = (df['num_lotes_listado'] != df['lotes']).sum()
print(f'Iguales   : {iguales}')
print(f'Distintos : {distintos}')

print("---------------------------------")
# Ver ejemplos de los 2 juntos
print('Muestra de las 2 columnas:')
df[['num_lotes_listado', 'lotes']].head(5)


=== LOTES ===
¿num_lotes_listado == lotes?
Iguales   : 1537
Distintos : 0
---------------------------------
Muestra de las 2 columnas:


,num_lotes_listado,lotes
0,1,1
1,1,1
2,1,1
3,1,1
4,1,1


In [9]:
# ====== Verificar las 2 columnas posiblemente iguales ('fecha_conclusion_listado', 'fecha_conclusion')=======

print('=== FECHA CONCLUSION ===')
print(f'fecha_conclusion_listado == fecha_conclusion?')
iguales = (df['fecha_conclusion_listado'] == df['fecha_conclusion']).sum()
distintos = (df['fecha_conclusion_listado'] != df['fecha_conclusion']).sum()
print(f'Iguales   : {iguales}')
print(f'Distintos : {distintos}')

print("---------------------------------")
# Ver ejemplos de los 2 juntos
print('Muestra de las 2 columnas:')
df[['fecha_conclusion_listado', 'fecha_conclusion']].head(7)



=== FECHA CONCLUSION ===
fecha_conclusion_listado == fecha_conclusion?
Iguales   : 0
Distintos : 1537
---------------------------------
Muestra de las 2 columnas:


,fecha_conclusion_listado,fecha_conclusion
0,NaN,NaN
1,01/12/2025 a las 18:00:00,2025-12-01
2,01/12/2025 a las 18:00:00,2025-12-01
3,01/12/2025 a las 18:00:00,2025-12-01
4,01/12/2025 a las 18:00:00,2025-12-01
5,01/12/2025 a las 18:00:00,2025-12-01
6,01/12/2025 a las 18:00:00,2025-12-01


### **1.3 Elimando columnas duplicadas.** 

In [10]:
# Columnas a eliminar y motivo
columnas_eliminar = [
    'id_sub',                  # idéntica a 'referencia'
    'referencia_det',          # idéntica a 'referencia'
    'num_lotes_listado',       # idéntica a 'lotes'
    'fecha_conclusion_listado' # duplicada en formato sucio '01/12/2025 a las 18:00:00'  (texto con hora)
]

df = df.drop(columns=columnas_eliminar).copy()

print(f'Columnas eliminadas: {columnas_eliminar}')
print(f'Columnas restantes : {df.shape[1]}')
print(f'Shape actual       : {df.shape}')

Columnas eliminadas: ['id_sub', 'referencia_det', 'num_lotes_listado', 'fecha_conclusion_listado']
Columnas restantes : 19
Shape actual       : (1537, 19)


In [11]:
print('Columnas actuales:')
for i, col in enumerate(df.columns):
    print(f'{i}.{col}')

Columnas actuales:
0.referencia
1.organismo
2.expediente
3.estado
4.descripcion
5.url_detalle
6.tipo_subasta
7.cantidad_reclamada_eur
8.lotes
9.anuncio_boe
10.valor_subasta_eur
11.tasacion_eur
12.puja_minima_eur
13.tramo_pujas_eur
14.deposito_eur
15.fecha_inicio
16.fecha_conclusion
17.cuenta_expediente
18.forma_adjudicacion


### **1.4 Eliminando columnas que no aportan al análisis**

In [12]:
columnas_eliminar = [
    'url_detalle', # URL
    'anuncio_boe'  # código interno del BOE
]

df = df.drop(columns= columnas_eliminar).copy()
print(f'Columnas a eliminar: {columnas_eliminar}')
print(f'Shape actual:        {df.shape}')

Columnas a eliminar: ['url_detalle', 'anuncio_boe']
Shape actual:        (1537, 17)


In [13]:
df.head()

,referencia,organismo,expediente,estado,descripcion,tipo_subasta,cantidad_reclamada_eur,lotes,valor_subasta_eur,tasacion_eur,puja_minima_eur,tramo_pujas_eur,deposito_eur,fecha_inicio,fecha_conclusion,cuenta_expediente,forma_adjudicacion
0,SUB-RC-2025-3800100125018,Agencia Tributaria Canaria - Las Palmas de Gra...,NaN,Suspendida,URBANA: Número veinticinco. -Unidad de Alojami...,RECAUDACIÓN TRIBUTARIA,315821.33,1,105744.66,105744.66,NaN,1500.0,5287.23,NaN,NaN,NaN,NaN
1,SUB-AT-2025-24R4686001676,U.R. SUBASTAS VALENCIA - VALENCIA (AEAT),NaN,Finalizada y depósitos con reserva devueltos,"VIVIENDA. CL/ SIENA, 37 BJ J. 28027 - MADRID (...",AGENCIA TRIBUTARIA,NaN,1,55110.30,311336.41,NaN,1000.0,2755.51,2025-11-11,2025-12-01,NaN,NaN
2,SUB-AT-2025-25R0886001398,U.R. SUBASTAS CATALUÑA - BARCELONA (AEAT),NaN,Finalizada y depósitos con reserva devueltos,SOLAR. UR FONT DEL BOSC 108 MEDIONA BARCELONA,AGENCIA TRIBUTARIA,NaN,1,47400.00,79000.00,4740.00,1000.0,2370.00,2025-11-11,2025-12-01,NaN,NaN
3,SUB-AT-2025-25R0886001399,U.R. SUBASTAS CATALUÑA - BARCELONA (AEAT),NaN,Finalizada y depósitos con reserva devueltos,SOLAR. UR FONT DEL BOSC . C/ BELLOC 131 MEDION...,AGENCIA TRIBUTARIA,NaN,1,49200.00,82000.00,4920.00,1000.0,2460.00,2025-11-11,2025-12-01,NaN,NaN
4,SUB-AT-2025-25R0886001455,U.R. SUBASTAS CATALUÑA - BARCELONA (AEAT),NaN,Finalizada y depósitos con reserva devueltos,100% PLENO DOMINIO DE GARAJE. PD/ DEL CAMP . 2...,AGENCIA TRIBUTARIA,NaN,1,15264.48,15264.48,1526.45,500.0,763.22,2025-11-11,2025-12-01,NaN,NaN


### **1.5 Corrigiendo tipo de datos**

In [21]:
#==== Fechas a tipo datetime=======
df['fecha_inicio']= pd.to_datetime(df['fecha_inicio'], errors= 'coerce') #coerce --> si no puede convertir un valor a fecha pone NaT
df['fecha_conclusion']= pd.to_datetime(df['fecha_conclusion'], errors= 'coerce')

#=====Verificamos cambios ======
print(df.dtypes)

referencia                           str
organismo                            str
expediente                           str
estado                               str
descripcion                          str
tipo_subasta                         str
cantidad_reclamada_eur           float64
lotes                              int64
valor_subasta_eur                float64
tasacion_eur                     float64
puja_minima_eur                  float64
tramo_pujas_eur                  float64
deposito_eur                     float64
fecha_inicio              datetime64[us]
fecha_conclusion          datetime64[us]
cuenta_expediente                    str
forma_adjudicacion                   str
dtype: object


In [22]:
df.head()

,referencia,organismo,expediente,estado,descripcion,tipo_subasta,cantidad_reclamada_eur,lotes,valor_subasta_eur,tasacion_eur,puja_minima_eur,tramo_pujas_eur,deposito_eur,fecha_inicio,fecha_conclusion,cuenta_expediente,forma_adjudicacion
0,SUB-RC-2025-3800100125018,Agencia Tributaria Canaria - Las Palmas de Gra...,NaN,Suspendida,URBANA: Número veinticinco. -Unidad de Alojami...,RECAUDACIÓN TRIBUTARIA,315821.33,1,105744.66,105744.66,NaN,1500.0,5287.23,NaT,NaT,NaN,NaN
1,SUB-AT-2025-24R4686001676,U.R. SUBASTAS VALENCIA - VALENCIA (AEAT),NaN,Finalizada y depósitos con reserva devueltos,"VIVIENDA. CL/ SIENA, 37 BJ J. 28027 - MADRID (...",AGENCIA TRIBUTARIA,NaN,1,55110.30,311336.41,NaN,1000.0,2755.51,2025-11-11,2025-12-01,NaN,NaN
2,SUB-AT-2025-25R0886001398,U.R. SUBASTAS CATALUÑA - BARCELONA (AEAT),NaN,Finalizada y depósitos con reserva devueltos,SOLAR. UR FONT DEL BOSC 108 MEDIONA BARCELONA,AGENCIA TRIBUTARIA,NaN,1,47400.00,79000.00,4740.00,1000.0,2370.00,2025-11-11,2025-12-01,NaN,NaN
3,SUB-AT-2025-25R0886001399,U.R. SUBASTAS CATALUÑA - BARCELONA (AEAT),NaN,Finalizada y depósitos con reserva devueltos,SOLAR. UR FONT DEL BOSC . C/ BELLOC 131 MEDION...,AGENCIA TRIBUTARIA,NaN,1,49200.00,82000.00,4920.00,1000.0,2460.00,2025-11-11,2025-12-01,NaN,NaN
4,SUB-AT-2025-25R0886001455,U.R. SUBASTAS CATALUÑA - BARCELONA (AEAT),NaN,Finalizada y depósitos con reserva devueltos,100% PLENO DOMINIO DE GARAJE. PD/ DEL CAMP . 2...,AGENCIA TRIBUTARIA,NaN,1,15264.48,15264.48,1526.45,500.0,763.22,2025-11-11,2025-12-01,NaN,NaN


### **1.6 Analizando valores nulos**

In [26]:
#============Nulos por columna=========
nulos = (df.isnull().sum() / len(df) * 100).round(2)
nulos = nulos[nulos > 0].sort_values(ascending=False)

print('Columnas con valores nulos (%):')
print(nulos.to_string())

Columnas con valores nulos (%):
forma_adjudicacion        85.30
puja_minima_eur           77.81
tramo_pujas_eur           28.82
expediente                22.97
cuenta_expediente         22.97
cantidad_reclamada_eur    16.27
valor_subasta_eur         15.22
deposito_eur              14.70
tasacion_eur              14.70
fecha_conclusion           6.70
fecha_inicio               6.70
descripcion                2.73


**Explicación de decisión de los valores nulos**

| Columna | Nulos % | Decisión | Motivo |
|---------|---------|----------|--------|
| ***forma_adjudicacion*** | ***85%*** |  ~~Eliminar~~ | ***Solo aparece en subastas con varios lotes*** |
| puja_minima_eur | 78% |  Conservar | "Sin puja mínima" es información válida |
| tramo_pujas_eur | 29% |  Conservar | Útil para análisis económico |
| ***expediente*** | ***23%*** |  ~~Eliminar~~ | ***Solo judicial, no aporta al análisis*** |
| ***cuenta_expediente*** | ***23%*** |  ~~Eliminar~~ | ***Solo judicial, no aporta al análisis*** |
| cantidad_reclamada_eur | 16% |  Conservar | Útil para comparar judicial vs AEAT |
| valor_subasta_eur | 15% |  Conservar | Campo clave para hipótesis |
| tasacion_eur | 15% |  Conservar | Campo clave para hipótesis |
| deposito_eur | 15% |  Conservar | Útil para análisis económico |
| fecha_conclusion | 7% |  Conservar | Campo temporal importante |
| fecha_inicio | 7% |  Conservar | Campo temporal importante |
| descripcion | 3% |  Conservar | Descripción del bien |

In [ ]:
#============== Eliminando columnas con registros nulos que no aportan ===============
columnas_eliminar = ['forma_adjudicacion', 'expediente', 'cuenta_expediente']
df = df.drop(columns= columnas_eliminar).copy()

#======= Verificando  =====================
print(f'Colummas eliminadas:`{columnas_eliminar}')
print(f'Shape actual:         {df.shape} ')

Colummas eliminadas:`['forma_adjudicacion', 'expediente', 'cuenta_expediente']
Shape actual:         (1537, 14) 


In [26]:
print(f'\nColumnas actuales:')
for i, col in enumerate(df.columns):
    print(f'  {i:2d}. {col}')


Columnas actuales:
   0. referencia
   1. organismo
   2. estado
   3. descripcion
   4. tipo_subasta
   5. cantidad_reclamada_eur
   6. lotes
   7. valor_subasta_eur
   8. tasacion_eur
   9. puja_minima_eur
  10. tramo_pujas_eur
  11. deposito_eur
  12. fecha_inicio
  13. fecha_conclusion


### **1.7 Calculando descuento_pct**
Mide cuánto por debajo de la tasación sale el precio de subasta.<br>

**- Cálculo:** Generalmente se calcula como:<br>
        **descuento_pct = (tasacion_eur - valor_subasta_eur) / tasacion_eur * 100**

**-Ejemplo:** <br>
tasacion_eur      = 100.000 €<br>
valor_subasta_eur =  60.000 €<br>
descuento_pct = (100.000 - 60.000) / 100.000 * 100 = 40%<br>

--> El inmueble sale a subasta con un 40% de descuento sobre su valor de mercado.




In [ ]:
#======= mascara para obtener registros que tengan los 3 campos datos mayores a 0
mask = (
    df['tasacion_eur'].notna() &
    df['valor_subasta_eur'].notna() &
    (df['tasacion_eur'] > 0)
)

#=========== Crea la columna descuento_pct con los datos calculados
df.loc[mask, 'descuento_pct'] = (
    (df.loc[mask, 'tasacion_eur'] - df.loc[mask, 'valor_subasta_eur']) / df.loc[mask, 'tasacion_eur'] * 100
).round(2)


descuento_pct calculado
Con descuento: 583
Sin descuento: 954


In [38]:
print(f'descuento_pct calculado')
print(f'Con descuento  : {mask.sum()}')
print(f'Sin descuento  : {df["descuento_pct"].isna().sum()}')
print(f'\nEstadísticas:')
print(df['descuento_pct'].describe().round(2))

descuento_pct calculado
Con descuento  : 583
Sin descuento  : 954

Estadísticas:
count        583.00
mean       -1729.36
std        41563.08
min     -1003515.83
25%            0.00
50%            0.00
75%            1.12
max           98.84
Name: descuento_pct, dtype: float64


In [75]:
# ============ En la descripcion estadistica se observa la media negativa =========
# Ver los casos extremos negativos
print('=== VALORES EXTRAÑOS ===')
print('\nTop 5 descuentos más negativos:')
print(df[['referencia', 'tasacion_eur', 'valor_subasta_eur', 'descuento_pct']]
      .sort_values('descuento_pct').head(5))

print('\nTop 5 descuentos más altos:')
print(df[['referencia', 'tasacion_eur', 'valor_subasta_eur', 'descuento_pct']]
      .sort_values('descuento_pct', ascending= False).head(5))


=== VALORES EXTRAÑOS ===

Top 5 descuentos más negativos:
              referencia  tasacion_eur  valor_subasta_eur  descuento_pct
1242  SUB-JA-2025-254826        120.00          1204339.0    -1003515.83
223   SUB-JV-2025-253747     182794.64         18279464.0       -9900.00
1161  SUB-JA-2025-254305     137124.18           170360.5         -24.24
650   SUB-JV-2025-254338   15885229.30         16287700.0          -2.53
1469  SUB-JA-2025-251876     184764.00           184764.0           0.00

Top 5 descuentos más altos:
                     referencia  tasacion_eur  valor_subasta_eur  \
916   SUB-AT-2025-25R5086001073      21067.72             244.91   
1266         SUB-JA-2025-255112       6405.20             405.20   
1360         SUB-JA-2025-253367      41252.25            2923.11   
249          SUB-JA-2025-254262      63288.87            6564.46   
632          SUB-JA-2025-254288     550000.00           61875.00   

      descuento_pct  
916           98.84  
1266          93.67  


In [77]:

print('¿Cuántos tienen tasacion_eur == 0?')
print((df['tasacion_eur'] == 0).count())

print('\n¿Cuántos tienen descuento negativo?')
print((df['descuento_pct'] < 0).sum())

print('\n¿Cuántos tienen descuento == 0?')
print((df['descuento_pct'] == 0).sum())

¿Cuántos tienen tasacion_eur == 0?
1537

¿Cuántos tienen descuento negativo?
4

¿Cuántos tienen descuento == 0?
430


### **1.9 Anomalías detectadas en los datos**

Durante la exploración se detectaron registros donde el `valor_subasta_eur` supera a la `tasacion_eur`. 
Verificando directamente en el portal del BOE se confirmó que **estos errores provienen del origen** (datos introducidos incorrectamente por los juzgados), no del scraping.

Ejemplos encontrados:
- SUB-JA-2025-254826: Tasación = 120€, Valor subasta = 1.204.339€
- SUB-JA-2025-254305: Tasación = 137.124€, Valor subasta = 170.360€

**Decisión:** estos registros se excluyen del análisis de 
descuentos por no ser representativos, pero se conservan 
en el dataset original.

### **1.10 Filtrar regitros con valores coherentes**
Condiciones para un registro válido:
1. tasacion_eur > valor_subasta_eur (la tasación debe ser mayor)
2. descuento_pct entre 0% y 100%    (descuento lógico)
3. tasacion_eur > 1000              (evitar tasaciones simbólicas como 120€)

In [78]:
mask_validos = (
    (df['tasacion_eur'] > df['valor_subasta_eur']) &
    (df['descuento_pct'] >= 0) &
    (df['descuento_pct'] <= 100) &
    (df['tasacion_eur'] > 1000)
)

print(f'Registros originales :{len(df)}')
print(f'Registros válidos    :{mask_validos.sum()}')
print(f'Registros descartados:{(~mask_validos).sum()}')

# Ver los descartados para entender qué son
print(f'\nRegistros descartados por motivo:')
print(f'  tasacion < valor_subasta : {(df["tasacion_eur"] < df["valor_subasta_eur"]).sum()}')
print(f'  descuento_pct == 0       : {(df["descuento_pct"] == 0).sum()}')
print(f'  tasacion <= 1000€        : {(df["tasacion_eur"] <= 1000).sum()}')
print(f'  descuento negativo       : {(df["descuento_pct"] < 0).sum()}')

Registros originales :1537
Registros válidos    :149
Registros descartados:1388

Registros descartados por motivo:
  tasacion < valor_subasta : 724
  descuento_pct == 0       : 430
  tasacion <= 1000€        : 722
  descuento negativo       : 4


### **1.8 Calculando la duración de la subasta**

In [31]:
# Cuántos días duró cada subasta
mask_fechas = df['fecha_inicio'].notna() & df['fecha_conclusion'].notna() #notna()filas no vacias

df.loc[mask_fechas, 'duracion_dias'] = (
    df.loc[mask_fechas, 'fecha_conclusion'] - 
    df.loc[mask_fechas, 'fecha_inicio']
).dt.days

print(f'duracion_dias calculado')
print(f'\nEstadísticas:')
print(df['duracion_dias'].describe())

duracion_dias calculado

Estadísticas:
count    1434.000000
mean       20.797071
std         8.930918
min        20.000000
25%        20.000000
50%        20.000000
75%        20.000000
max       220.000000
Name: duracion_dias, dtype: float64


=== VALORES EXTRAÑOS ===

Top 7 descuentos más negativos:
                     referencia  tasacion_eur  valor_subasta_eur  descuento_pct
1242         SUB-JA-2025-254826        120.00         1204339.00    -1003515.83
223          SUB-JV-2025-253747     182794.64        18279464.00       -9900.00
1161         SUB-JA-2025-254305     137124.18          170360.50         -24.24
650          SUB-JV-2025-254338   15885229.30        16287700.00          -2.53
1469         SUB-JA-2025-251876     184764.00          184764.00           0.00
1473  SUB-RC-2025-0026I20250024     177552.00          177552.00           0.00
1477  SUB-RC-2025-0026I20250140      54809.52           54809.52           0.00

Top 7 descuentos más altos:
                     referencia  tasacion_eur  valor_subasta_eur  descuento_pct
916   SUB-AT-2025-25R5086001073      21067.72             244.91          98.84
1266         SUB-JA-2025-255112       6405.20             405.20          93.67
1360         SUB-JA-2025-253367  

In [42]:
# Ver cuántos registros tienen valores razonables (entre -100 y 100)
razonables = df[(df['descuento_pct'] >= -100) & (df['descuento_pct'] <= 100)]
print(f'Registros con descuento entre -100% y 100%: {len(razonables)}')
print(razonables['descuento_pct'].describe().round(2))

Registros con descuento entre -100% y 100%: 581
count    581.00
mean       8.95
std       19.02
min      -24.24
25%        0.00
50%        0.00
75%        1.31
max       98.84
Name: descuento_pct, dtype: float64


In [79]:
print('=== DIAGNÓSTICO TASACION ===')
print(f'tasacion_eur == 0    : {(df["tasacion_eur"] == 0).sum()}')
print(f'tasacion_eur nula    : {df["tasacion_eur"].isna().sum()}')
print(f'tasacion_eur > 0     : {(df["tasacion_eur"] > 0).sum()}')

print('\n=== DIAGNÓSTICO VALOR SUBASTA ===')
print(f'valor_subasta nulo   : {df["valor_subasta_eur"].isna().sum()}')
print(f'valor_subasta == 0   : {(df["valor_subasta_eur"] == 0).sum()}')
print(f'valor_subasta > 0    : {(df["valor_subasta_eur"] > 0).sum()}')

print('\n=== POR TIPO DE SUBASTA ===')
print(df.groupby('tipo_subasta')[['tasacion_eur', 'valor_subasta_eur']].agg(
    lambda x: x.isna().sum()
).rename(columns={'tasacion_eur': 'nulos_tasacion', 
                  'valor_subasta_eur': 'nulos_valor'}))

=== DIAGNÓSTICO TASACION ===
tasacion_eur == 0    : 720
tasacion_eur nula    : 226
tasacion_eur > 0     : 591

=== DIAGNÓSTICO VALOR SUBASTA ===
valor_subasta nulo   : 234
valor_subasta == 0   : 0
valor_subasta > 0    : 1303

=== POR TIPO DE SUBASTA ===
                            nulos_tasacion  nulos_valor
tipo_subasta                                           
AGENCIA TRIBUTARIA                       0            0
JUDICIAL CONCURSAL                       7            7
JUDICIAL EN VÍA DE APREMIO             192          192
JUDICIAL VOLUNTARIA                      8            8
NOTARIAL HIPOTECARIA                     1            3
OTRAS SUBASTAS NOTARIALES                0            6
RECAUDACIÓN TRIBUTARIA                  18           18


In [82]:
# Ver los valores raw de tasacion donde sale 0
# Necesitamos volver al CSV original para ver cómo llegaron
print('=== VERIFICAR FORMATO NÚMEROS ===')

# Recargar el CSV original para ver los valores crudos
df_raw = pd.read_csv('./data/subastas_inmuebles_dic2025.csv')

# Ver algunos registros donde tasacion_eur == 0
print('Registros con tasacion == 0 en df actual:')
mask_cero = df['tasacion_eur'] == 0
print(df_raw[mask_cero.values][['referencia', 'tasacion_eur']].head(10).to_string())

=== VERIFICAR FORMATO NÚMEROS ===
Registros con tasacion == 0 en df actual:
            referencia  tasacion_eur
19  SUB-JA-2018-112984           0.0
20  SUB-JA-2022-198527           0.0
21  SUB-JA-2024-224678           0.0
22  SUB-JA-2024-226691           0.0
23  SUB-JA-2024-228932           0.0
25  SUB-JA-2025-241010           0.0
26  SUB-JA-2025-249410           0.0
27  SUB-JA-2025-249586           0.0
28  SUB-JA-2025-250208           0.0
29  SUB-JA-2025-250760           0.0


In [83]:
print(df_raw[df_raw['referencia'] == 'SUB-JA-2025-241010']['url_detalle'].values[0])

https://subastas.boe.es/detalleSubasta.php?idSub=SUB-JA-2025-241010&idBus=b1dsZHcyK2F3Z2VwRFVqaXQ1OGVPbGZZWnlDUzY0ekcvQTAzTDNNSks0RTJWL0h4UXRTRW1hdnlIRDJuMzJjMVgxYks0MUpxak5iWWpQNnZGaGhSMGVmUTNyUnpkS0JvWUtiSm56a1IxcjdYQXRpY2E2dHVFNnNtMXNoWDFvak1EeXhZVTZNQ3JCUlk0WVptemtmbHlLT21sbkF0TVpFOHRXYnp3Lzhhc0xzVmREWWNzVllPekhpTFp0NjRzQXJIaFdLY3h4WjNsc0tCR2JBOXN1d01lWXp0aG1iUjZTWTlmNkF3bHdGSFRRa2tCeG5WZnJFUzl2cmljMldsd1Q5VE1MWFJqVmF5VElCM2xud2s1UE5tckVPLzNHVjV1Vy8wbVpMdnpCRUdWeGxrUHFMODlmOVZVN09pVVEvcldOZGI,--50


In [90]:
df[df['referencia'] == 'SUB-JA-2025-251767']

,referencia,organismo,estado,descripcion,tipo_subasta,cantidad_reclamada_eur,lotes,valor_subasta_eur,tasacion_eur,puja_minima_eur,tramo_pujas_eur,deposito_eur,fecha_inicio,fecha_conclusion,descuento_pct,duracion_dias
89,SUB-JA-2025-251767,JUZGADO 1 INSTANCIA 11 - VALLADOLID,Pendiente de finalización y devolución de depó...,50% DEL PLENO DOMINIO POR TITULO DE COMPRAVENT...,JUDICIAL EN VÍA DE APREMIO,19235.4,1,5355.47,40833.5,NaN,107.11,267.77,2025-11-11,2025-12-01,86.88,20.0


In [85]:
# ¿Cuántos registros tiene la AEAT?
df_aeat = df[df['tipo_subasta'] == 'AGENCIA TRIBUTARIA'].copy()

print(f'Total registros AEAT         : {len(df_aeat)}')
print(f'Con tasacion > 0             : {(df_aeat["tasacion_eur"] > 0).sum()}')
print(f'Con valor_subasta > 0        : {(df_aeat["valor_subasta_eur"] > 0).sum()}')
print(f'Con ambos campos completos   : {(df_aeat["tasacion_eur"].notna() & df_aeat["valor_subasta_eur"].notna()).sum()}')
print(f'\nPorcentaje del total original: {len(df_aeat)/len(df)*100:.1f}%')
print(f'\nTipos de subasta en dataset completo:')
print(df['tipo_subasta'].value_counts())

Total registros AEAT         : 239
Con tasacion > 0             : 239
Con valor_subasta > 0        : 239
Con ambos campos completos   : 239

Porcentaje del total original: 15.5%

Tipos de subasta en dataset completo:
tipo_subasta
JUDICIAL EN VÍA DE APREMIO    1111
AGENCIA TRIBUTARIA             239
RECAUDACIÓN TRIBUTARIA         104
JUDICIAL VOLUNTARIA             52
JUDICIAL CONCURSAL              21
OTRAS SUBASTAS NOTARIALES        7
NOTARIAL HIPOTECARIA             3
Name: count, dtype: int64


In [86]:
# Verificar RECAUDACIÓN TRIBUTARIA
df_rec = df[df['tipo_subasta'] == 'RECAUDACIÓN TRIBUTARIA'].copy()

print(f'=== RECAUDACIÓN TRIBUTARIA ===')
print(f'Total registros              : {len(df_rec)}')
print(f'Con tasacion > 0             : {(df_rec["tasacion_eur"] > 0).sum()}')
print(f'Con valor_subasta > 0        : {(df_rec["valor_subasta_eur"] > 0).sum()}')
print(f'Con ambos campos completos   : {(df_rec["tasacion_eur"].notna() & df_rec["valor_subasta_eur"].notna()).sum()}')

=== RECAUDACIÓN TRIBUTARIA ===
Total registros              : 104
Con tasacion > 0             : 86
Con valor_subasta > 0        : 86
Con ambos campos completos   : 86
